In [1]:
!pip install nnunetv2 scipy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 kB 8.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.7/100.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Faz 4 — nnU-Net v2 Eğitimi (Fold 0)

Bu notebook yalnızca eğitim adımını içerir. Ön koşullar:
- `nndet/nifti/` altında NIfTI hacimleri mevcut olmalı
- `nndet/nnunet_raw/Dataset100_Abdomen/` hazır olmalı
- `nndet/nnunet_preprocessed/` plan+preprocess tamamlanmış olmalı

Yukarıdakileri hazırlamak için önce `Faz4_nnDetection_3B_1fold-colab1.ipynb` Section 1–5'i çalıştırın.

In [2]:
import os, sys, shutil, subprocess, json
import pandas as pd
from pathlib import Path
from typing import List

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
NND_ROOT = Path('/content/drive/MyDrive/Abdomen/outputs/nndet')


NNUNET_RAW          = NND_ROOT / "nnunet_raw"
NNUNET_PREPROCESSED = NND_ROOT / "nnunet_preprocessed"
NNUNET_RESULTS      = NND_ROOT / "nnunet_results"

DATASET_ID   = 100
DATASET_NAME = "Abdomen"

os.environ["nnUNet_raw"]          = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"]      = str(NNUNET_RESULTS)
os.environ["OMP_NUM_THREADS"]     = "1"

import sysconfig as _sc
def _nnunet_cmd(name: str) -> str:
    p = shutil.which(name)
    if p: return p
    for d in [_sc.get_path("scripts"), str(Path(sys.executable).parent),
               "/usr/local/bin", "/opt/conda/bin"]:
        c = Path(d) / name
        if c.exists(): return str(c)
    return name

print(f"nnUNet_raw         : {os.environ['nnUNet_raw']}")
print(f"nnUNet_preprocessed: {os.environ['nnUNet_preprocessed']}")
print(f"nnUNet_results     : {os.environ['nnUNet_results']}")

_cmd_train = _nnunet_cmd("nnUNetv2_train")
print(f"nnUNetv2_train     : {_cmd_train}  (var={Path(_cmd_train).exists()})")

nnUNet_raw         : /content/drive/MyDrive/Abdomen/outputs/nndet/nnunet_raw
nnUNet_preprocessed: /content/drive/MyDrive/Abdomen/outputs/nndet/nnunet_preprocessed
nnUNet_results     : /content/drive/MyDrive/Abdomen/outputs/nndet/nnunet_results
nnUNetv2_train     : /usr/local/bin/nnUNetv2_train  (var=True)


In [12]:
SUPER_CLASSES: List[str] = [
    "acute_cholecystitis",        # 0 → label 1
    "kidney_ureter_stone",        # 1 → label 2
    "acute_pancreatitis",         # 2 → label 3
    "aortic_aneurysm_dissection", # 3 → label 4
    "acute_appendicitis",         # 4 → label 5
    "acute_diverticulitis",       # 5 → label 6
]


## 6. Fold 0 Eğitim

In [13]:
import torch

# ── GPU kontrolü ──────────────────────────────────────────────────────────
_has_cuda = torch.cuda.is_available()
_device   = "cuda" if _has_cuda else "cpu"
print(f"Cihaz: {_device}")

Cihaz: cuda


In [14]:

if _has_cuda:
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

if not _has_cuda:
    print("\n⚠️  CUDA bulunamadı — eğitim GPU gerektirir.")
    print("Bu notebook Colab'da GPU runtime ile çalıştırılmalıdır:")
    print("  Runtime → Change runtime type → T4 GPU (veya A100)")
    raise SystemExit("GPU olmadan eğitim başlatılmadı.")

GPU   : Tesla T4
VRAM  : 15.6 GB


In [15]:

# ── nnUNetv2_train ────────────────────────────────────────────────────────
print(f"\nnnU-Net v2 eğitimi — fold 0, 3d_fullres, ~8-15 saat (T4)...")
print(f"Çıktılar: {NNUNET_RESULTS}")



nnU-Net v2 eğitimi — fold 0, 3d_fullres, ~8-15 saat (T4)...
Çıktılar: /content/drive/MyDrive/Abdomen/outputs/nndet/nnunet_results


In [16]:
import subprocess
import os

_cmd = _nnunet_cmd("nnUNetv2_train")
# --npz: softmax olasılık haritalarını kaydeder (güven skoru için)
r = subprocess.run(
    [_cmd, str(DATASET_ID), "3d_fullres", "0", "--npz"],
    env={**os.environ}, capture_output=True, text=True
)

print("--- stdout ---")
print(r.stdout)
print("--- stderr ---")
print(r.stderr)

print("Eğitim çıktı kodu:", r.returncode)
if r.returncode != 0:
    print("⚠️  Eğitim hata kodu döndürüyor (bkz. yukarı)")

--- stdout ---

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-05-25 07:53:43.158867: Using torch.compile...
2026-05-25 07:53:48.899504: do_dummy_2d_data_aug: False
2026-05-25 07:53:48.951479: Using splits from existing split file: /content/drive/MyDrive/Abdomen/outputs/nndet/nnunet_preprocessed/Dataset100_Abdomen/splits_final.json
2026-